In [1]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.impute import SimpleImputer
import joblib
import warnings
warnings.filterwarnings("ignore")

sys.path.append("../src")
from feature_extraction.audio_features import extract_features

In [2]:
dataset = []
ravdess_path = "../data/ravdess"

for actor_folder in os.listdir(ravdess_path):
    actor_path = os.path.join(ravdess_path, actor_folder)
    if not os.path.isdir(actor_path):
        continue
    for filename in os.listdir(actor_path):
        if not filename.endswith(".wav"):
            continue
        parts = filename.replace(".wav", "").split("-")
        if len(parts) < 3:
            continue
        emotion = int(parts[2])
        # Depression proxy labels
        # 0 = healthy (neutral=1, calm=2, happy=3)
        # 1 = depressed (sad=4, fearful=6, disgust=7)
        if emotion in [1, 2, 3]:
            label = 0
        elif emotion in [4, 6, 7]:
            label = 1
        else:
            continue  # skip angry and surprised
        filepath = os.path.join(actor_path, filename)
        try:
            features = extract_features(filepath)
            features["file"]  = filepath
            features["label"] = label
            dataset.append(features)
        except Exception as e:
            print(f"Failed {filename}: {e}")

df = pd.DataFrame(dataset)
print(f"Dataset shape: {df.shape}")
print(df["label"].value_counts())

Dataset shape: (1056, 21)
label
1    576
0    480
Name: count, dtype: int64


In [3]:
df.to_csv("../data/voice_features/depression_dataset.csv", index=False)
print("Saved depression_dataset.csv")

Saved depression_dataset.csv


In [4]:
feature_cols = [
    "pitch", "spectral_centroid", "zcr",
    "jitter", "shimmer", "hnr",
    "mfcc_1","mfcc_2","mfcc_3","mfcc_4","mfcc_5","mfcc_6","mfcc_7",
    "mfcc_8","mfcc_9","mfcc_10","mfcc_11","mfcc_12","mfcc_13"
]

X = df[feature_cols]
y = df["label"]

imputer = SimpleImputer(strategy="mean")
X_imp = imputer.fit_transform(X)

# ── 5-Fold Cross Validation ──────────────────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model_cv = RandomForestClassifier(n_estimators=200, random_state=42)

cv_acc  = cross_val_score(model_cv, X_imp, y, cv=skf, scoring="accuracy")
cv_auc  = cross_val_score(model_cv, X_imp, y, cv=skf, scoring="roc_auc")

print("5-Fold Cross Validation Results:")
print(f"  Accuracy : {cv_acc.mean():.4f} +/- {cv_acc.std():.4f}")
print(f"  AUROC    : {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}")

# ── Final model on full data ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_imp, y, test_size=0.2, random_state=42, stratify=y
)
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f"\nHeld-out Test Set Results:")
print(f"  Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"  AUROC    : {roc_auc_score(y_test, y_prob):.4f}")
print(classification_report(y_test, y_pred, target_names=["Healthy", "Depressed"]))

5-Fold Cross Validation Results:
  Accuracy : 0.7633 +/- 0.0295
  AUROC    : 0.8555 +/- 0.0320

Held-out Test Set Results:
  Accuracy : 0.8113
  AUROC    : 0.8892
              precision    recall  f1-score   support

     Healthy       0.84      0.72      0.78        96
   Depressed       0.79      0.89      0.84       116

    accuracy                           0.81       212
   macro avg       0.82      0.80      0.81       212
weighted avg       0.81      0.81      0.81       212



In [5]:
os.makedirs("../models", exist_ok=True)
joblib.dump(model,   "../models/depression_model.pkl")
joblib.dump(imputer, "../models/depression_imputer.pkl")
print("Saved depression_model.pkl")
print("Saved depression_imputer.pkl")

Saved depression_model.pkl
Saved depression_imputer.pkl
